#### Fit Initial Model

In [2]:
import pandas as pd

In [3]:
movies_enriched = pd.read_parquet("../data/movie_with_features.parquet")
movie_ratings_w_tmbd = pd.read_parquet("../data/movielens_ratings_w_tmdbId.parquet")
content_modeling_data = pd.read_parquet("../data/content_modeling_data.parquet")

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1,2)
)

movie_matrix = tfidf.fit_transform(
    content_modeling_data["content_text"]
)

In [5]:
feature_names = tfidf.get_feature_names_out()

feature_names[10000:10050]

array(['keyword_adoring keyword_amused', 'keyword_adult_animation',
       'keyword_adult_animation keyword_anime',
       'keyword_adult_child_friendship', 'keyword_adult_humor',
       'keyword_adulterous_wife', 'keyword_adultery',
       'keyword_adultery keyword_infidelity', 'keyword_adventure',
       'keyword_adventurer', 'keyword_advertising',
       'keyword_advertising_executive', 'keyword_affectation',
       'keyword_afghanistan', 'keyword_africa',
       'keyword_african_american', 'keyword_aftercreditsstinger',
       'keyword_aftercreditsstinger keyword_duringcreditsstinger',
       'keyword_afterlife', 'keyword_age', 'keyword_age gap_relationship',
       'keyword_age_difference', 'keyword_agent', 'keyword_aggressive',
       'keyword_aging', 'keyword_agriculture', 'keyword_aids',
       'keyword_aimless_youth', 'keyword_airplane',
       'keyword_airplane_crash', 'keyword_airplane_hijacking',
       'keyword_airport', 'keyword_alabama', 'keyword_alaska',
       'keyword

In [6]:
tf_idf_matrix = tfidf.fit_transform(
    content_modeling_data["content_text"]).toarray()


In [10]:
import scipy.sparse as sp
sp.save_npz("../models/tfidf_matrix.npz", movie_matrix)
content_modeling_data[["tmdbId"]].to_parquet("../models/tfidf_tmdb_ids.parquet", index=False)

In [11]:
print(movie_matrix.shape, len(content_modeling_data))

(87426, 20000) 87426


In [ ]:
# LOCATION: notebooks/05_content_based_model.ipynb

from pathlib import Path
from scipy.sparse import save_npz

# Assumes the notebook is running from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().parent
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# This must be the same dataframe, in the same row order,
# used to create tf_idf_matrix.
content_movie_catalog = movies_enriched.reset_index(drop=True).copy()

assert tf_idf_matrix.shape[0] == len(content_movie_catalog)

save_npz(models_dir / "tf_idf_matrix.npz", tf_idf_matrix.tocsr())

content_movie_catalog.to_parquet(
    models_dir / "content_movie_catalog.parquet",
    index=False
)

print("Saved matrix:", tf_idf_matrix.shape)
print("Saved catalog:", content_movie_catalog.shape)

In [ ]:
def content_based_recommendations(title, similar_movies):

    recommended_movies = []

    idx = movies_enriched[movies_enriched["title"] == title].index[0]
    score_series = pd.Series(similar_movies[idx]).sort_values(ascending=False)

    top10_indexes = list(score_series.iloc[1:11].index)

    for i in top10_indexes:
        recommended_movies.append(
            {
                "title": movies_enriched.iloc[i]["title"],
                "score": score_series.iloc[i]
            }
        )
    return recommended_movies

In [ ]:
content_based_recommendations("The Dark Knight", similar_movies)